In [168]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

In [169]:
df = pd.read_csv("../data/supply_chain_disruption_recovery.csv")

df = df.dropna()

print(df.shape)

(100000, 15)


In [ ]:
# for production impact

features_impact = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
    "response_type",
    "response_time_days"
]

y = df["production_impact_pct"]

In [ ]:
features_recovery = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
    "response_type",
    "response_time_days"
]

y = df["full_recovery_days"]

In [ ]:
features_revenue = [
    "production_impact_pct",
    "full_recovery_days",
    "disruption_severity",
    "response_time_days",
    "industry",
    "supplier_tier",
    "supplier_size"
]

y = df["revenue_loss_usd"]

In [189]:
print(df[[
    "production_impact_pct",
    "full_recovery_days",
    "revenue_loss_usd"
]].corr(numeric_only=True))

                       production_impact_pct  full_recovery_days  \
production_impact_pct               1.000000            0.417026   
full_recovery_days                  0.417026            1.000000   
revenue_loss_usd                    0.254029            0.090520   

                       revenue_loss_usd  
production_impact_pct          0.254029  
full_recovery_days             0.090520  
revenue_loss_usd               1.000000  


In [193]:
from sklearn.ensemble import HistGradientBoostingRegressor

impact_features = features_impact

impact_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

impact_cat_cols = [c for c in impact_cat_cols if c in impact_features]

impact_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), impact_cat_cols)
], remainder="passthrough")

X = df[impact_features]
y = df["production_impact_pct"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

impact_model = Pipeline([
    ("preprocessor", impact_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

impact_model.fit(X_train, y_train)
pred = impact_model.predict(X_test)

print("production Impact MAE:", mean_absolute_error(y_test, pred))
print("production Impact R2:", r2_score(y_test, pred))

joblib.dump(impact_model, "impact_model.pkl")

production Impact MAE: 7.776373290085465
production Impact R2: 0.7006283589105324


['impact_model.pkl']

In [194]:
recovery_features = features_recovery

recovery_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

recovery_cat_cols = [c for c in recovery_cat_cols if c in recovery_features]

recovery_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), recovery_cat_cols)
], remainder="passthrough")

X = df[recovery_features]
y = np.log1p(df["full_recovery_days"])   # IMPORTANT FIX

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

recovery_model = Pipeline([
    ("preprocessor", recovery_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=600,
        random_state=42
    ))
])

recovery_model.fit(X_train, y_train)

pred_log = recovery_model.predict(X_test)

pred = np.expm1(pred_log)
y_true = np.expm1(y_test)

print("RECOVERY MAE:", mean_absolute_error(y_true, pred))
print("RECOVERY R2:", r2_score(y_true, pred))

joblib.dump(recovery_model, "recovery_model.pkl")

RECOVERY MAE: 30.511168882442266
RECOVERY R2: 0.3622215348063025


['recovery_model.pkl']

In [195]:
revenue_features = features_revenue

revenue_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

revenue_cat_cols = [c for c in revenue_cat_cols if c in revenue_features]

revenue_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), revenue_cat_cols)
], remainder="passthrough")

q99 = df["revenue_loss_usd"].quantile(0.99)
df_rev = df[df["revenue_loss_usd"] <= q99]

X = df_rev[revenue_features]
y = np.log1p(df_rev["revenue_loss_usd"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

revenue_model = Pipeline([
    ("preprocessor", revenue_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=10,
        max_iter=700,
        random_state=42
    ))
])

revenue_model.fit(X_train, y_train)

pred_log = revenue_model.predict(X_test)

pred = np.expm1(pred_log)
y_true = np.expm1(y_test)

print("REVENUE MAE:", mean_absolute_error(y_true, pred))
print("REVENUE R2:", r2_score(y_true, pred))

joblib.dump(revenue_model, "revenue_model.pkl")

REVENUE MAE: 1142532.8404197202
REVENUE R2: 0.46678572613947


['revenue_model.pkl']

In [196]:
print(df["revenue_loss_usd"].describe())

count    1.000000e+05
mean     2.522812e+06
std      5.533130e+06
min      1.000000e+04
25%      3.016020e+05
50%      8.754662e+05
75%      2.492971e+06
max      3.731310e+08
Name: revenue_loss_usd, dtype: float64


In [197]:
sample = pd.DataFrame(
    {
        "disruption_type":["Geopolitical"],
        "industry":["Automotive"],
        "supplier_tier":["Tier 1"],
        "supplier_region":["East Asia"],
        "supplier_size":["Large"],
        "has_backup_supplier":["No"],
        "disruption_severity":[5],
        "response_type":["Alternative Supplier"],
        "response_time_days":[10]
    }
)

In [198]:
predicted_impact = impact_model.predict(
    sample
)[0]

In [199]:
predicted_recovery = recovery_model.predict(
    sample
)[0]

In [201]:
revenue_input = pd.DataFrame(
    {
        "production_impact_pct":[predicted_impact],
        "full_recovery_days":[predicted_recovery],
        "disruption_severity":[5],
        "response_time_days":[10],
         "supplier_size":["Large"],
          "industry":["Automotive"],
          "supplier_tier":["Tier 1"]
    }
)

predicted_revenue = revenue_model.predict(
    revenue_input
)[0]

In [202]:
def calculate_risk(
    impact,
    recovery,
    revenue
):

    impact_norm = impact / 100

    recovery_norm = min(
        recovery / 365,
        1
    )

    revenue_norm = min(
        revenue / 10000000,
        1
    )

    risk_score = (
        0.5 * impact_norm
        +
        0.3 * recovery_norm
        +
        0.2 * revenue_norm
    ) * 100

    return risk_score

In [203]:
def risk_level(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Medium"

    elif score < 75:
        return "High"

    else:
        return "Critical"

In [204]:
risk_score = calculate_risk(
    predicted_impact,
    predicted_recovery,
    predicted_revenue
)

risk = risk_level(
    risk_score
)

In [205]:
print(
    "Predicted Production Impact:",
    round(predicted_impact,2),
    "%"
)

print(
    "Predicted Recovery Time:",
    round(predicted_recovery,2),
    "days"
)

print(
    "Predicted Revenue Loss: $",
    round(predicted_revenue,2)
)

print(
    "Risk Score:",
    round(risk_score,2)
)

print(
    "Risk Level:",
    risk
)

Predicted Production Impact: 46.46 %
Predicted Recovery Time: 4.73 days
Predicted Revenue Loss: $ 15.56
Risk Score: 23.62
Risk Level: Low


In [ ]:

# for alternate supplier reccomendation , not complete , first creating synthetic supplier dataset.

suppliers = [
    ["SUP001","India","Tier 1","Large","Yes",4.8,5,92,85],
    ["SUP002","Vietnam","Tier 1","Large","Yes",4.6,6,89,82],
    ["SUP003","Mexico","Tier 1","Large","Yes",4.7,7,91,80],
    ["SUP004","Germany","Tier 1","Large","Yes",4.9,10,95,75],
    ["SUP005","Japan","Tier 1","Large","Yes",4.9,8,96,78],

    ["SUP006","India","Tier 2","Medium","Yes",4.5,4,90,88],
    ["SUP007","Vietnam","Tier 2","Medium","Yes",4.4,5,87,86],
    ["SUP008","Thailand","Tier 2","Medium","Yes",4.3,6,85,84],
    ["SUP009","Poland","Tier 2","Medium","Yes",4.5,8,91,79],
    ["SUP010","Turkey","Tier 2","Medium","No",4.1,9,82,90],

    ["SUP011","Brazil","Tier 3","Small","Yes",4.0,10,80,93],
    ["SUP012","Indonesia","Tier 3","Small","Yes",4.2,7,84,89],
    ["SUP013","Malaysia","Tier 3","Small","Yes",4.3,6,85,87],
    ["SUP014","South Korea","Tier 1","Large","Yes",4.8,5,94,81],
    ["SUP015","Singapore","Tier 1","Large","Yes",4.9,4,97,74],

    ["SUP016","France","Tier 1","Large","Yes",4.7,9,93,76],
    ["SUP017","Spain","Tier 2","Medium","Yes",4.4,8,88,83],
    ["SUP018","Canada","Tier 1","Large","Yes",4.8,6,95,77],
    ["SUP019","USA","Tier 1","Large","Yes",4.9,5,97,72],
    ["SUP020","Australia","Tier 2","Medium","Yes",4.5,7,90,80]
]

supplier_df = pd.DataFrame(
    suppliers,
    columns=[
        "supplier_id",
        "country",
        "supplier_tier",
        "supplier_size",
        "backup_capability",
        "supplier_rating",
        "lead_time_days",
        "quality_score",
        "cost_index"
    ]
)

supplier_df.to_csv(
    "supplier_dataset.csv",
    index=False
)

supplier_df.head()